In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

LAST_NAME = "Aldovino"
STUDENT_ID = "TUPM-26-4168"

id_digits = "".join(
    d for d in STUDENT_ID if d.isdigit()
)

assert id_digits, "Student ID must contain digits."

SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
NAME_LENGTH = len(LAST_NAME)

RANDOM_SEED = int(id_digits) % (2**32)

np.set_printoptions(precision=4, suppress=True)

# Locate dataset
WORK_DIR = Path.cwd()

if (WORK_DIR.parent / "data").exists():
    DATA_DIR = WORK_DIR.parent / "data"
else:
    DATA_DIR = WORK_DIR / "data"

OUTPUT_DIR = WORK_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

measurement_path = DATA_DIR / "secom.data"
label_path = DATA_DIR / "secom_labels.data"

raw_measurements = np.genfromtxt(
    measurement_path, dtype=float
)

labels = np.loadtxt(
    label_path, usecols=0, dtype=int
)

assert len(raw_measurements) == len(labels)

print("Student:", LAST_NAME)
print("Student ID:", STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Random Seed:", RANDOM_SEED)
print("Dataset Shape:", raw_measurements.shape)
print("Dataset Loaded Successfully!")

Student: Aldovino
Student ID: TUPM-26-4168
Seed Digit: 8
ID Sum: 27
Random Seed: 264168
Dataset Shape: (1567, 590)
Dataset Loaded Successfully!


In [2]:
missing = np.isnan(raw_measurements)

observed = (~missing).sum(axis=0)

all_missing = observed == 0

observed_min = np.where(
    missing, np.inf, raw_measurements
).min(axis=0)

observed_max = np.where(
    missing, -np.inf, raw_measurements
).max(axis=0)

constant = (
    (observed > 0) &
    (observed_min == observed_max)
)

usable_indices = np.flatnonzero(
    ~(all_missing | constant)
)

usable_features = [
    f"Feature_{i:03d}"
    for i in usable_indices
]

print("Total Features:", raw_measurements.shape[1])
print("Entirely Missing Features:", all_missing.sum())
print("Constant Features:", constant.sum())
print("Usable Features:", len(usable_indices))

Total Features: 590
Entirely Missing Features: 0
Constant Features: 116
Usable Features: 474


In [4]:
# [CELL 1] EXERCISE 1: Vectorized Measurement Quality Analysis
rng = np.random.default_rng(RANDOM_SEED)

feature_count = 5 + (SEED_DIGIT % 3)

selected_positions = rng.choice(
    len(usable_indices),
    size=feature_count,
    replace=False
)

selected_indices = usable_indices[selected_positions]

selected_features = [
    f"Feature_{i:03d}"
    for i in selected_indices
]

selected_data = raw_measurements[:, selected_indices]

print("Selected Features:", selected_features)
print("Number of Selected Features:", len(selected_features))

Selected Features: ['Feature_441', 'Feature_452', 'Feature_393', 'Feature_520', 'Feature_107', 'Feature_211', 'Feature_383']
Number of Selected Features: 7


In [5]:
# [CELL 2] Calculate numerical statistics
feature_means = np.nanmean(selected_data, axis=0)

feature_medians = np.nanmedian(selected_data, axis=0)

feature_sds = np.nanstd(
    selected_data, axis=0, ddof=1
)

missing_percentages = (
    np.isnan(selected_data).mean(axis=0) * 100
)

print("Feature Means:", feature_means)
print("Feature Medians:", feature_medians)
print("Sample Standard Deviations:", feature_sds)
print("Missing Percentages:", missing_percentages)

Feature Means: [ 0.8021  5.3468  0.134   2.696  -0.0018  0.0568  0.541 ]
Feature Medians: [0.8076 5.2714 0.1339 2.221  0.     0.0532 0.3726]
Sample Standard Deviations: [0.1842 0.9192 0.0384 5.7024 0.0875 0.025  1.341 ]
Missing Percentages: [ 0.0638  0.0638  0.      0.      0.3829  1.5316 64.9649]


In [6]:
# [CELL 3] Calculate robust standardized values
selected_medians = np.nanmedian(
    selected_data, axis=0
)

absolute_deviations = np.abs(
    selected_data - selected_medians
)

mad = np.nanmedian(
    absolute_deviations, axis=0
)

safe_mad = np.where(mad > 0, mad, 1.0)

robust_z = (
    0.6745 *
    (selected_data - selected_medians) /
    safe_mad
)

robust_z = np.where(
    np.isnan(selected_data), np.nan, robust_z
)

flags = np.abs(robust_z) > 3.5

flagged_counts = np.sum(flags, axis=0)

observed_counts = np.sum(
    ~np.isnan(selected_data), axis=0
)

flagged_percentages = (
    flagged_counts / observed_counts * 100
)

print("MAD:", mad)
print("Flagged Measurements:", flagged_counts)
print("Flagged Percentages:", flagged_percentages)
print("Total Flagged Observations:", flags.sum())

MAD: [0.1256 0.5717 0.0276 0.6795 0.049  0.0103 0.1022]
Flagged Measurements: [ 0  6  1 35 24 42 58]
Flagged Percentages: [ 0.      0.3831  0.0638  2.2336  1.5375  2.722  10.5647]
Total Flagged Observations: 166


In [7]:
# [CELL 4] Rank features and export the CSV
quality_summary = pd.DataFrame({
    "Feature": selected_features,
    "Mean": feature_means,
    "Median": feature_medians,
    "Sample_SD": feature_sds,
    "Missing_Percentage": missing_percentages,
    "MAD": mad,
    "Flagged_Count": flagged_counts,
    "Flagged_Percentage": flagged_percentages
})

quality_summary = quality_summary.sort_values(
    "Flagged_Percentage",
    ascending=False
).reset_index(drop=True)

quality_summary.to_csv(
    OUTPUT_DIR / "measurement_quality.csv",
    index=False
)

highest_missing = missing_percentages.max()

highest_flagged_feature = quality_summary.iloc[0]["Feature"]

total_flagged = int(flags.sum())

print(quality_summary.round(4))
print("Highest Missing Percentage:", highest_missing)
print("Feature with Highest Flagged Percentage:",
      highest_flagged_feature)
print("Total Flagged Observations:", total_flagged)
print("Vectorized Results Verified:",
      np.array_equal(flags, np.abs(robust_z) > 3.5))

       Feature    Mean  Median  Sample_SD  Missing_Percentage     MAD  \
0  Feature_383  0.5410  0.3726     1.3410             64.9649  0.1022   
1  Feature_211  0.0568  0.0532     0.0250              1.5316  0.0103   
2  Feature_520  2.6960  2.2210     5.7024              0.0000  0.6795   
3  Feature_107 -0.0018  0.0000     0.0875              0.3829  0.0490   
4  Feature_452  5.3468  5.2714     0.9192              0.0638  0.5717   
5  Feature_393  0.1340  0.1339     0.0384              0.0000  0.0276   
6  Feature_441  0.8021  0.8076     0.1842              0.0638  0.1256   

   Flagged_Count  Flagged_Percentage  
0             58             10.5647  
1             42              2.7220  
2             35              2.2336  
3             24              1.5375  
4              6              0.3831  
5              1              0.0638  
6              0              0.0000  
Highest Missing Percentage: 64.96490108487556
Feature with Highest Flagged Percentage: Feature_383
Tota

In [8]:
# [CELL 4] EXERCISE 2: Mobile Device Usage Profiling
n_features = raw_measurements.shape[1]

feature_names = [
    f"Feature_{i:03d}"
    for i in range(n_features)
]

df = pd.DataFrame(
    raw_measurements.copy(),
    columns=feature_names
)

df.insert(
    0,
    "Record_ID",
    np.arange(len(df))
)

df["Result"] = labels

df["Outcome"] = np.where(
    labels == -1, "Pass", "Fail"
)

print(df.head())
print("\nOutcome Counts:")
print(df["Outcome"].value_counts())

   Record_ID  Feature_000  Feature_001  Feature_002  Feature_003  Feature_004  \
0          0      3030.93      2564.00    2187.7333    1411.1265       1.3602   
1          1      3095.78      2465.14    2230.4222    1463.6606       0.8294   
2          2      2932.61      2559.94    2186.4111    1698.0172       1.5102   
3          3      2988.72      2479.90    2199.0333     909.7926       1.3204   
4          4      3032.24      2502.87    2233.3667    1326.5200       1.5334   

   Feature_005  Feature_006  Feature_007  Feature_008  ...  Feature_582  \
0        100.0      97.6133       0.1242       1.5005  ...       0.5005   
1        100.0     102.3433       0.1247       1.4966  ...       0.5019   
2        100.0      95.4878       0.1241       1.4436  ...       0.4958   
3        100.0     104.2367       0.1217       1.4882  ...       0.4990   
4        100.0     100.3967       0.1235       1.5031  ...       0.4800   

   Feature_583  Feature_584  Feature_585  Feature_586  Feature

In [9]:
# [CELL 5] Create a balanced sample
pass_count = (labels == -1).sum()
fail_count = (labels == 1).sum()

balanced_size = min(
    20 + ID_SUM,
    pass_count,
    fail_count
)

rng = np.random.default_rng(RANDOM_SEED)

pass_positions = rng.choice(
    np.flatnonzero(labels == -1),
    size=balanced_size,
    replace=False
)

fail_positions = rng.choice(
    np.flatnonzero(labels == 1),
    size=balanced_size,
    replace=False
)

balanced_positions = np.concatenate([
    pass_positions,
    fail_positions
])

balanced_df = df.iloc[
    balanced_positions
].copy()

print("Balanced Sample Size per Group:", balanced_size)
print("\nBalanced Outcome Counts:")
print(balanced_df["Outcome"].value_counts())

Balanced Sample Size per Group: 47

Balanced Outcome Counts:
Outcome
Pass    47
Fail    47
Name: count, dtype: int64


In [10]:
# [CELL 6] Select five features and apply median imputation
rng = np.random.default_rng(RANDOM_SEED)

selected_positions = rng.choice(
    len(usable_indices),
    size=5,
    replace=False
)

selected_indices = usable_indices[selected_positions]

selected_features = [
    f"Feature_{i:03d}"
    for i in selected_indices
]

full_medians = np.nanmedian(
    raw_measurements[:, selected_indices],
    axis=0
)

balanced_values = balanced_df[
    selected_features
].to_numpy(dtype=float)

balanced_values = np.where(
    np.isnan(balanced_values),
    full_medians,
    balanced_values
)

balanced_df[selected_features] = balanced_values

print("Selected Features:", selected_features)
print("Remaining Missing Values:",
      balanced_df[selected_features].isna().sum().sum())

Selected Features: ['Feature_454', 'Feature_406', 'Feature_108', 'Feature_442', 'Feature_384']
Remaining Missing Values: 0


In [11]:
# [CELL 7] Calculate grouped statistics
grouped_stats = balanced_df.groupby(
    "Outcome"
)[selected_features].agg(
    ["count", "mean", "median", "std"]
)

print("Grouped Statistical Analysis:")
print(grouped_stats.round(4))

Grouped Statistical Analysis:
        Feature_454                         Feature_406                  \
              count    mean  median     std       count    mean  median   
Outcome                                                                   
Fail             47  7.2481  6.8395  2.0519          47  7.0247  6.1702   
Pass             47  7.8654  7.1457  2.8112          47  6.8632  6.5530   

                Feature_108                         Feature_442          \
            std       count    mean  median     std       count    mean   
Outcome                                                                   
Fail     2.8842          47 -0.0085 -0.0001  0.0992          47  1.2253   
Pass     2.0938          47 -0.0262  0.0000  0.0950          47  1.4590   

                        Feature_384                          
         median     std       count    mean  median     std  
Outcome                                                      
Fail     1.1879  0.5806         

In [12]:
# [CELL 8] Calculate standardized mean differences
comparison = []

for feature in selected_features:

    pass_values = balanced_df.loc[
        balanced_df["Outcome"] == "Pass",
        feature
    ].to_numpy()

    fail_values = balanced_df.loc[
        balanced_df["Outcome"] == "Fail",
        feature
    ].to_numpy()

    pass_mean = pass_values.mean()
    fail_mean = fail_values.mean()

    pass_sd = pass_values.std(ddof=1)
    fail_sd = fail_values.std(ddof=1)

    pooled_sd = np.sqrt(
        (
            (len(pass_values) - 1) * pass_sd**2
            +
            (len(fail_values) - 1) * fail_sd**2
        )
        /
        (
            len(pass_values) + len(fail_values) - 2
        )
    )

    mean_difference = fail_mean - pass_mean

    absolute_difference = abs(mean_difference)

    standardized_difference = (
        absolute_difference / pooled_sd
        if pooled_sd > 0 else np.nan
    )

    comparison.append({
        "Feature": feature,
        "Pass_Mean": pass_mean,
        "Fail_Mean": fail_mean,
        "Absolute_Mean_Difference": absolute_difference,
        "Pooled_SD": pooled_sd,
        "Absolute_Standardized_Mean_Difference":
            standardized_difference
    })

comparison_df = pd.DataFrame(comparison)

comparison_df = comparison_df.sort_values(
    "Absolute_Standardized_Mean_Difference",
    ascending=False,
    na_position="last"
).reset_index(drop=True)

comparison_df["Rank"] = np.arange(
    1, len(comparison_df) + 1
)

print(comparison_df.round(4))

       Feature  Pass_Mean  Fail_Mean  Absolute_Mean_Difference  Pooled_SD  \
0  Feature_442     1.4590     1.2253                    0.2337     0.6148   
1  Feature_454     7.8654     7.2481                    0.6173     2.4610   
2  Feature_384     1.0786     1.1265                    0.0479     0.2332   
3  Feature_108    -0.0262    -0.0085                    0.0177     0.0971   
4  Feature_406     6.8632     7.0247                    0.1615     2.5202   

   Absolute_Standardized_Mean_Difference  Rank  
0                                 0.3801     1  
1                                 0.2508     2  
2                                 0.2053     3  
3                                 0.1820     4  
4                                 0.0641     5  


In [13]:
# [CELL 9] Verify NumPy and Pandas results and export files
numpy_match = True

for feature in selected_features:

    for outcome in ["Pass", "Fail"]:

        values = balanced_df.loc[
            balanced_df["Outcome"] == outcome,
            feature
        ].to_numpy()

        numpy_mean = np.mean(values)
        numpy_median = np.median(values)
        numpy_sd = np.std(values, ddof=1)

        pandas_values = grouped_stats.loc[outcome, feature]

        numpy_match = numpy_match and np.allclose(
            [
                numpy_mean,
                numpy_median,
                numpy_sd
            ],
            [
                pandas_values["mean"],
                pandas_values["median"],
                pandas_values["std"]
            ]
        )

balanced_df.to_csv(
    OUTPUT_DIR / "balanced_sample.csv",
    index=False
)

comparison_df.to_csv(
    OUTPUT_DIR / "feature_comparison.csv",
    index=False
)

print("Balanced Sample Size:", len(balanced_df))
print("Selected Features:", selected_features)
print("Highest-Ranked Feature:",
      comparison_df.iloc[0]["Feature"])
print("Absolute Standardized Mean Difference:",
      comparison_df.iloc[0][
          "Absolute_Standardized_Mean_Difference"
      ])
print("NumPy-Pandas Statistics Match:", numpy_match)
print("Two CSV files exported successfully.")

Balanced Sample Size: 94
Selected Features: ['Feature_454', 'Feature_406', 'Feature_108', 'Feature_442', 'Feature_384']
Highest-Ranked Feature: Feature_442
Absolute Standardized Mean Difference: 0.38013953776611287
NumPy-Pandas Statistics Match: True
Two CSV files exported successfully.


In [14]:
# [CELL 10] EXCERCISE 3: Statistical Robustness and Sensitivity Analysis
eligible_indices = []

for index in usable_indices:

    pass_observed = np.sum(
        (~np.isnan(raw_measurements[labels == -1, index]))
    )

    fail_observed = np.sum(
        (~np.isnan(raw_measurements[labels == 1, index]))
    )

    if pass_observed >= 10 and fail_observed >= 10:
        eligible_indices.append(index)

rng = np.random.default_rng(RANDOM_SEED)

selected_indices = rng.choice(
    eligible_indices,
    size=5,
    replace=False
)

selected_features = [
    f"Feature_{i:03d}"
    for i in selected_indices
]

print("Selected Features:", selected_features)
print("Number of Eligible Features:", len(eligible_indices))

Selected Features: ['Feature_455', 'Feature_407', 'Feature_107', 'Feature_443', 'Feature_385']
Number of Eligible Features: 470


In [15]:
# [CELL 11] Prepare the two data strategies
selected_data = raw_measurements[:, selected_indices]

full_medians = np.nanmedian(
    selected_data, axis=0
)

prepared_median = np.where(
    np.isnan(selected_data),
    full_medians,
    selected_data
)

print("Selected Features:", selected_features)
print("Available-Case Strategy: Missing values excluded")
print("Median-Imputation Strategy: Missing values replaced")

Selected Features: ['Feature_455', 'Feature_407', 'Feature_107', 'Feature_443', 'Feature_385']
Available-Case Strategy: Missing values excluded
Median-Imputation Strategy: Missing values replaced


In [16]:
# [CELL 12] Calculate group mean differences and bootstrap confidence intervals
BOOTSTRAP_REPETITIONS = 200 + 10 * SEED_DIGIT

def run_robustness_analysis():

    rng = np.random.default_rng(RANDOM_SEED)

    results = []

    for j, feature in enumerate(selected_features):

        original_values = selected_data[:, j]

        for strategy in [
            "Available Case",
            "Median Imputation"
        ]:

            if strategy == "Available Case":

                pass_values = original_values[
                    (labels == -1) &
                    ~np.isnan(original_values)
                ]

                fail_values = original_values[
                    (labels == 1) &
                    ~np.isnan(original_values)
                ]

            else:

                pass_values = prepared_median[
                    labels == -1, j
                ]

                fail_values = prepared_median[
                    labels == 1, j
                ]

            observed_difference = (
                fail_values.mean() - pass_values.mean()
            )

            # Bootstrap sampling independently by outcome
            pass_draws = rng.choice(
                pass_values,
                size=(
                    BOOTSTRAP_REPETITIONS,
                    len(pass_values)
                ),
                replace=True
            )

            fail_draws = rng.choice(
                fail_values,
                size=(
                    BOOTSTRAP_REPETITIONS,
                    len(fail_values)
                ),
                replace=True
            )

            bootstrap_differences = (
                fail_draws.mean(axis=1)
                -
                pass_draws.mean(axis=1)
            )

            bootstrap_mean = bootstrap_differences.mean()

            lower_ci, upper_ci = np.percentile(
                bootstrap_differences,
                [2.5, 97.5]
            )

            results.append({
                "Feature": feature,
                "Strategy": strategy,
                "Observed_Mean_Difference":
                    observed_difference,
                "Bootstrap_Mean_Difference":
                    bootstrap_mean,
                "CI_Lower": lower_ci,
                "CI_Upper": upper_ci
            })

    return pd.DataFrame(results)


robustness_df = run_robustness_analysis()

print("Bootstrap Repetitions:", BOOTSTRAP_REPETITIONS)
print(robustness_df.round(4))

Bootstrap Repetitions: 280
       Feature           Strategy  Observed_Mean_Difference  \
0  Feature_455     Available Case                    0.2120   
1  Feature_455  Median Imputation                    0.2119   
2  Feature_407     Available Case                    0.0310   
3  Feature_407  Median Imputation                    0.0309   
4  Feature_107     Available Case                   -0.0053   
5  Feature_107  Median Imputation                   -0.0054   
6  Feature_443     Available Case                    0.0212   
7  Feature_443  Median Imputation                    0.0212   
8  Feature_385     Available Case                    0.0022   
9  Feature_385  Median Imputation                    0.0022   

   Bootstrap_Mean_Difference  CI_Lower  CI_Upper  
0                     0.2134    0.0468    0.4094  
1                     0.2140    0.0413    0.3863  
2                     0.0281   -0.0336    0.0891  
3                     0.0346   -0.0358    0.0982  
4                    -0.

In [17]:
# [CELL 13] Compare the two strategies
available = (
    robustness_df[
        robustness_df["Strategy"] == "Available Case"
    ]
    .set_index("Feature")
)

imputed = (
    robustness_df[
        robustness_df["Strategy"] == "Median Imputation"
    ]
    .set_index("Feature")
)

comparison = pd.DataFrame({
    "Available_Case_Difference":
        available["Observed_Mean_Difference"],

    "Median_Imputation_Difference":
        imputed["Observed_Mean_Difference"]
})

comparison["Absolute_Change"] = abs(
    comparison["Available_Case_Difference"]
    -
    comparison["Median_Imputation_Difference"]
)

comparison["Available_Case_Rank"] = (
    comparison["Available_Case_Difference"]
    .abs()
    .rank(ascending=False, method="min")
)

comparison["Median_Imputation_Rank"] = (
    comparison["Median_Imputation_Difference"]
    .abs()
    .rank(ascending=False, method="min")
)

comparison["Changed_Sign"] = (
    np.sign(comparison["Available_Case_Difference"])
    !=
    np.sign(comparison["Median_Imputation_Difference"])
)

comparison["Changed_Ranking"] = (
    comparison["Available_Case_Rank"]
    !=
    comparison["Median_Imputation_Rank"]
)

print("Sensitivity Comparison:")
print(comparison.round(4))

Sensitivity Comparison:
             Available_Case_Difference  Median_Imputation_Difference  \
Feature                                                                
Feature_455                     0.2120                        0.2119   
Feature_407                     0.0310                        0.0309   
Feature_107                    -0.0053                       -0.0054   
Feature_443                     0.0212                        0.0212   
Feature_385                     0.0022                        0.0022   

             Absolute_Change  Available_Case_Rank  Median_Imputation_Rank  \
Feature                                                                     
Feature_455           0.0001                  1.0                     1.0   
Feature_407           0.0001                  2.0                     2.0   
Feature_107           0.0000                  4.0                     4.0   
Feature_443           0.0000                  3.0                     3.0   
Feature_3

In [21]:
# [CELL 14] Verify reproducibility
repeated_results = run_robustness_analysis()

columns_to_check = [
    "Observed_Mean_Difference",
    "Bootstrap_Mean_Difference",
    "CI_Lower",
    "CI_Upper"
]

reproducibility_verified = np.allclose(
    robustness_df[columns_to_check].to_numpy(),
    repeated_results[columns_to_check].to_numpy(),
    rtol=1e-10,
    atol=1e-10
)

print("Reproducibility Verified:",
      reproducibility_verified)

Reproducibility Verified: True


In [22]:
# [cell 15] Generate the comparison CSV and sensitivity report
largest_change = comparison["Absolute_Change"].max()

changed_sign_features = comparison.index[
    comparison["Changed_Sign"]
].tolist()

changed_rank_features = comparison.index[
    comparison["Changed_Ranking"]
].tolist()

robustness_df.to_csv(
    OUTPUT_DIR / "robustness_comparison.csv",
    index=False
)

report = f"""
STATISTICAL ROBUSTNESS AND SENSITIVITY ANALYSIS

Selected Features:
{', '.join(selected_features)}

Bootstrap Repetitions:
{BOOTSTRAP_REPETITIONS}

Largest Change in Mean Difference:
{largest_change:.4f}

Features with Changed Difference Sign:
{changed_sign_features}

Features with Changed Ranking:
{changed_rank_features}

Reproducibility Verified:
{reproducibility_verified}

Interpretation:
The analysis compared available-case analysis and median imputation.
The bootstrap method was used to estimate mean differences and
95% confidence intervals.

Changes in mean differences, confidence intervals, or feature
rankings show how data preparation methods can affect statistical
results. These findings should be interpreted carefully.
"""

(OUTPUT_DIR / "sensitivity_report.txt").write_text(
    report,
    encoding="utf-8"
)

print(report)
print("Required output files generated.")


STATISTICAL ROBUSTNESS AND SENSITIVITY ANALYSIS

Selected Features:
Feature_455, Feature_407, Feature_107, Feature_443, Feature_385

Bootstrap Repetitions:
280

Largest Change in Mean Difference:
0.0001

Features with Changed Difference Sign:
[]

Features with Changed Ranking:
[]

Reproducibility Verified:
True

Interpretation:
The analysis compared available-case analysis and median imputation.
The bootstrap method was used to estimate mean differences and
95% confidence intervals.

Changes in mean differences, confidence intervals, or feature
rankings show how data preparation methods can affect statistical
results. These findings should be interpreted carefully.

Required output files generated.


In [23]:
# [cell 16] Check all generated files
CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "data").exists():
    PROJECT_DIR = CURRENT_DIR
    EXERCISE_DIR = (
        PROJECT_DIR / "Lab7_NumPy_Pandas_Additional_Exercises"
    )
else:
    PROJECT_DIR = CURRENT_DIR.parent
    EXERCISE_DIR = CURRENT_DIR

DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = EXERCISE_DIR / "output"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

measurement_path = DATA_DIR / "secom.data"
label_path = DATA_DIR / "secom_labels.data"

In [24]:
print("===== EXERCISE 1 =====")
print("Selected Features:", len(selected_features))
print("Highest Missing Percentage:",
      f"{highest_missing:.4f}%")
print("Highest Flagged Feature:",
      highest_flagged_feature)
print("Total Flagged Observations:",
      total_flagged)
print("Vectorized Results Verified: True")

print("\n===== EXERCISE 2 =====")
print("Balanced Sample Size:", len(balanced_df))
print("Selected Features:", selected_features)
print("Highest-Ranked Feature:",
      comparison_df.iloc[0]["Feature"])
print("NumPy-Pandas Statistics Match:", numpy_match)

print("\n===== EXERCISE 3 =====")
print("Selected Features:", selected_features)
print("Bootstrap Repetitions:",
      BOOTSTRAP_REPETITIONS)
print("Largest Change in Mean Difference:",
      largest_change)
print("Changed Sign Features:", changed_sign_features)
print("Changed Ranking Features:", changed_rank_features)
print("Reproducibility Verified:",
      reproducibility_verified)

===== EXERCISE 1 =====
Selected Features: 5
Highest Missing Percentage: 64.9649%
Highest Flagged Feature: Feature_383
Total Flagged Observations: 166
Vectorized Results Verified: True

===== EXERCISE 2 =====
Balanced Sample Size: 94
Selected Features: ['Feature_455', 'Feature_407', 'Feature_107', 'Feature_443', 'Feature_385']
Highest-Ranked Feature: Feature_442
NumPy-Pandas Statistics Match: True

===== EXERCISE 3 =====
Selected Features: ['Feature_455', 'Feature_407', 'Feature_107', 'Feature_443', 'Feature_385']
Bootstrap Repetitions: 280
Largest Change in Mean Difference: 6.968277240826382e-05
Changed Sign Features: []
Changed Ranking Features: []
Reproducibility Verified: True
